In [1]:
import numpy as np
from gnuradio import gr
from gnuradio import blocks
from gnuradio import analog

ModuleNotFoundError: No module named 'gnuradio'

In [ ]:
class TwoFSKGenerator(gr.top_block):
    def __init__(self, samp_rate, freq_low, freq_high, bits_per_symbol):
        gr.top_block.__init__(self, "2FSK Generator")

        self.samp_rate = samp_rate
        self.freq_low = freq_low
        self.freq_high = freq_high
        self.bits_per_symbol = bits_per_symbol

        # 1. Source of your data (e.g., a stream of bits)
        self.source = blocks.vector_source_b([0, 1, 0, 1, 1, 0]) # Example bit stream

        # 2. Map bits to frequencies
        self.mapping = blocks.map({0: self.freq_low, 1: self.freq_high})

        # 3. Create a continuous wave oscillator
        self.oscillator = analog.sig_source_c(self.samp_rate, analog.GR_COS_WAVE, 0, 1)
        self.phase_acc = 0

        # 4. Frequency modulator (we'll implement this with a custom block)
        self.freq_mod = self.freq_modulator_cf(self.samp_rate)

        # 5. Optional: Audio sink to listen to the signal
        self.audio_sink = audio.sink(self.samp_rate, "")

        # Connect the blocks
        self.connect(self.source, self.mapping, self.freq_mod.freq)
        self.connect(self.oscillator, (self.freq_mod, 0))
        self.connect(self.freq_mod, self.audio_sink)

    class freq_modulator_cf(gr.basic_block):
        def __init__(self, samp_rate):
            gr.basic_block.__init__(
                self,
                name="Frequency Modulator",
                in_sig=[np.complex64, np.float32], # Input: Complex oscillator, Frequency
                out_sig=[np.complex64]          # Output: Modulated complex signal
            )
            self.samp_rate = samp_rate
            self.phase = 0

        def general_work(self, in_vec, out_vec):
            noutput_items = len(out_vec[0])
            in_osc = in_vec[0]
            in_freq = in_vec[1]
            out = out_vec[0]

            for i in range(noutput_items):
                frequency = in_freq[i % len(in_freq)] # Get the current frequency
                self.phase += 2 * np.pi * frequency / self.samp_rate
                out[i] = in_osc[i] * np.exp(1j * self.phase) # Apply frequency shift

            return noutput_items

In [ ]:
if __name__ == '__main__':
    try:
        samp_rate = 32000
        freq_low = 1000  # Lower frequency in Hz
        freq_high = 2000 # Higher frequency in Hz
        bits_per_symbol = 1

        tb = TwoFSKGenerator(samp_rate, freq_low, freq_high, bits_per_symbol)
        tb.start()
        tb.wait()
    except KeyboardInterrupt:
        pass
    finally:
        tb.stop()
        tb.wait()